# 1. FX Data

In [5]:
import os
import pandas as pd
import yfinance as yf

def download_fx(pair,start,end):
    df = yf.download("pair", start="start", end="end")
    df.info()
    os.makedirs("data/raw/fx", exist_ok=True)
    df.to_csv(f"data/raw/fx/{pair}.csv")

# 2. World Bank Data

In [13]:
import requests
import os

def download_wb(indicator, country, name):
    url = f"http://api.worldbank.org/v2/country/{country}/indicator/{indicator}?per_page=5000&format=csv"

    headers = {"User-Agent": "Mozilla/5.0"}
    res = requests.get(url, headers=headers)

    if res.status_code != 200:
        print(f"Failed to download data for {country} - {indicator}")
        print(res.text[:200])
        return None
    try:
        data = res.json()
    except:
        print(f"Failed to parse JSON for {country} - {indicator}")
        print(res.text[:500])
        return None
    
    if len(data) < 2:
        print(f"Unexpected JSON structure for {country} - {indicator}")
        return None
    
    df = pd.DataFrame(data[1])[['date', 'value']]
    df = df.dropna()
    df = df.sort_values('date')

    path = f"data/raw/macro/{country}.csv"
    os.makedirs(path, exist_ok=True)

    df.to_csv(f"{path}/{name}.csv", index=False)

    return df